Section 1:
The calendar file is separated by semicolons. Read the default way, it gives 3,068 × 2 columns with no error; with sep=";" it gives 3,068 × 9. The header line ends in a comma, so the last column is named eos_e, → every read must use sep=";", and Phase 3 must rename eos_e,. SOURCES.md wrongly says the header has no comma.

In [1]:
# importing neccessary libraries 

import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("..") / "data" / "raw"

# Recorded in docs/SOURCES.md at acquisition.
EXPECTED_MD5 = {
    "crop_calendar_gaul0.csv": "4430991cb6c58b82cea25cf5af503047",
    "crop_calendar_gaul1.csv": "86e9b947093c790f1af5949e79b8cb7e",
    "faostat_qcl_ghana.csv":   "e42769d1b60bcabe8dc8d42d2ef712b0",
    "bog_policy_rate.csv":     "dc7fa3634497158e9aa730643c204e95",
}
for name, expected in EXPECTED_MD5.items():
    actual = hashlib.md5((RAW / name).read_bytes()).hexdigest()
    print(name, "OK" if actual == expected else "CHANGED")
    assert actual == expected

print("pandas", pd.__version__)

crop_calendar_gaul0.csv OK
crop_calendar_gaul1.csv OK
faostat_qcl_ghana.csv OK
bog_policy_rate.csv OK
pandas 3.0.6


In [2]:
naive = pd.read_csv(RAW / "crop_calendar_gaul1.csv")
print("default read:", naive.shape)
print(list(naive.columns))
print()

default read: (3068, 2)
['asap0_id;name0_shr;asap1_id;name1_shr;crop_name;sos_s;sos_e;eos_s;eos_e', 'Unnamed: 1']



In [3]:
cal1 = pd.read_csv(RAW / "crop_calendar_gaul1.csv", sep=";")
print("with sep=';':", cal1.shape)
print(list(cal1.columns))

with sep=';': (3068, 9)
['asap0_id', 'name0_shr', 'asap1_id', 'name1_shr', 'crop_name', 'sos_s', 'sos_e', 'eos_s', 'eos_e,']


Section 2, trailing comma: Counted line endings in the raw file. 3,036 of 3,069 lines end in a comma, including the header; 33 data lines don't (8 of them Ghana). pandas stores all 3,068 values of eos_e, as text (e.g. "31,") → Phase 3 strips the comma and converts the column to numbers before any harvest-window calculation. SOURCES.md needs correcting.

In [4]:
cal1["name0_shr"]

0       Kyrgyzstan
1       Kyrgyzstan
2       Kyrgyzstan
3       Kyrgyzstan
4       Kyrgyzstan
           ...    
3063         Syria
3064         Syria
3065         Syria
3066         Syria
3067         Syria
Name: name0_shr, Length: 3068, dtype: str

In [5]:
# Read the raw file as bytes and split it into lines. Windows files end each line with \r\n.
lines = [ln for ln in (RAW / "crop_calendar_gaul1.csv").read_bytes().split(b"\r\n") if ln]
ends_with_comma = [ln.endswith(b",") for ln in lines]

print("lines (incl. header):  ", len(lines))
print("lines ending in comma: ", sum(ends_with_comma))
print("header ends in comma:  ", ends_with_comma[0])

no_comma = [ln.decode("utf-8") for ln, c in zip(lines, ends_with_comma) if not c]
print("data lines WITHOUT it: ", len(no_comma))
print("  by country:", pd.Series([ln.split(";")[1] for ln in no_comma]).value_counts().to_dict())
print()
print("types in last column:", cal1["eos_e,"].map(type).value_counts().to_dict())
print("sample values:", cal1["eos_e,"].head(3).tolist())

lines (incl. header):   3069
lines ending in comma:  3036
header ends in comma:   True
data lines WITHOUT it:  33
  by country: {"Côte d'Ivoire": 12, 'Ghana': 8, 'Guinea': 4, 'Nigeria': 3, 'Sierra Leone': 3, 'Uganda': 3}

types in last column: {<class 'str'>: 3068}
sample values: ['31,', '32,', '32,']


Section 3, calendar coverage: Checked which crop has planting and harvest dates in which region. Ghana has 34 rows across 10 regions (pre-2019 boundaries) and 7 crops. Upper East and Upper West have no maize row at all, although Wa and Bolga have maize prices almost every year; 5 regions lack main-season maize. Rice only covers the 3 northern regions and yams 4 southern ones → northern maize needs a borrowed or substitute calendar, chosen in Phase 3 and stated in the memo as an assumption.

In [6]:
gh = cal1[cal1["name0_shr"] == "Ghana"].copy()

# For LOOKING only: strip the comma so the last column becomes a number. The file is not touched.
gh["eos_e_num"] = gh["eos_e,"].str.rstrip(",").astype(int)

print("Ghana rows:", len(gh), "  regions:", gh["name1_shr"].nunique(), "  crops:", gh["crop_name"].nunique())

# One row per region, one column per crop.
pd.crosstab(gh["name1_shr"], gh["crop_name"])

Ghana rows: 34   regions: 10   crops: 7


crop_name,"Cassava (Year1, year2)",Maize (Second),Maize (South/Main),Millet,Rice (North),Sorghum,Yams
name1_shr,,,,,,,
Ashanti,1,1,0,0,0,0,0
Brong Ahafo,1,1,0,0,0,0,0
Central,1,1,1,0,0,0,1
Eastern,1,1,1,0,0,0,1
Greater Accra,1,1,1,0,0,0,0
Northern,1,1,0,1,1,1,0
Upper East,0,0,0,1,1,1,0
Upper West,0,0,0,1,1,1,0
Volta,1,1,1,0,0,0,1


Line 
Section 4, dekads and New Year: The calendar counts time in dekads (36 per year, about 10 days each). 7 Ghana rows have a harvest that crosses New Year: second-season maize in all 7 southern regions (December → January). In the north, millet, sorghum and rice share identical windows: planting May–June, harvest November (Upper West) or November–early December (Upper East) → Phase 3 must handle the year wrap explicitly when turning harvests into cash months, and the identical northern windows are a limitation to state. 


In [7]:
for c in ["sos_s", "sos_e", "eos_s", "eos_e_num"]:
    print(f"{c:10s} min {gh[c].min():2d}   max {gh[c].max():2d}")

# A harvest that ENDS in a lower dekad than it STARTS runs over New Year.
wraps = gh[gh["eos_e_num"] < gh["eos_s"]]
print("\nGhana rows whose harvest crosses New Year:", len(wraps))
wraps[["name1_shr", "crop_name", "eos_s", "eos_e_num"]]

sos_s      min  4   max 24
sos_e      min  9   max 28
eos_s      min 15   max 35
eos_e_num  min  1   max 36

Ghana rows whose harvest crosses New Year: 7


,name1_shr,crop_name,eos_s,eos_e_num
143,Eastern,Maize (Second),34,1
150,Ashanti,Maize (Second),35,1
152,Central,Maize (Second),34,1
156,Volta,Maize (Second),34,1
160,Brong Ahafo,Maize (Second),34,1
162,West,Maize (Second),34,2
166,Greater Accra,Maize (Second),35,1


In [8]:
# Dekad d falls in month (d - 1) // 3 + 1.  e.g. dekad 31 -> month 11 (November)
to_month = lambda d: pd.Timestamp(2019, (d - 1) // 3 + 1, 1).strftime("%b")

north = gh[gh["name1_shr"].isin(["Upper East", "Upper West"])][
    ["name1_shr", "crop_name", "sos_s", "sos_e", "eos_s", "eos_e_num"]].copy()
north["planting"] = north["sos_s"].map(to_month) + "-" + north["sos_e"].map(to_month)
north["harvest"]  = north["eos_s"].map(to_month) + "-" + north["eos_e_num"].map(to_month)
north

,name1_shr,crop_name,sos_s,sos_e,eos_s,eos_e_num,planting,harvest
146,Upper West,Millet,13,16,31,33,May-Jun,Nov-Nov
147,Upper West,Rice (North),13,16,31,33,May-Jun,Nov-Nov
148,Upper West,Sorghum,13,16,31,33,May-Jun,Nov-Nov
173,Upper East,Millet,12,17,31,34,Apr-Jun,Nov-Dec
174,Upper East,Rice (North),12,17,31,34,Apr-Jun,Nov-Dec
175,Upper East,Sorghum,12,17,31,34,Apr-Jun,Nov-Dec


line:
Section 5, region join: Compared calendar region names with price region names. The prices use capitals and the calendar doesn’t; once converted, 9 of 10 regions match. The calendar’s “West” doesn’t match the prices’ “WESTERN” → Phase 3 standardises the capitals and adds a WEST → WESTERN mapping, both logged. Without it, Western’s markets silently drop out of any calendar-to-price join.

In [9]:
price_regions = set(pd.read_csv(RAW / "wfp_food_prices_gha.csv", usecols=["admin1"])["admin1"].unique())
cal_regions = set(gh["name1_shr"].str.upper())

print("in calendar, not in prices:", sorted(cal_regions - price_regions))
print("in prices, not in calendar:", sorted(price_regions - cal_regions))

in calendar, not in prices: ['WEST']
in prices, not in calendar: ['WESTERN']


Section 6, FAOSTAT reading: The file starts with a 3-byte BOM (EF BB BF). pandas strips it automatically; the plain csv module keeps it (\ufeffDomain Code). Item Code (CPC) is an identifier: read as a number, 01520.01 becomes 1520.01 → always read the item code as text; the Phase 4 load must use utf-8-sig or strip the BOM. Phase 1’s note (“pandas needs utf-8-sig”) is corrected: the risk is in the database load, not in pandas.

In [10]:
import csv
print("first 3 bytes of file:", (RAW / "faostat_qcl_ghana.csv").read_bytes()[:3])

naive = pd.read_csv(RAW / "faostat_qcl_ghana.csv")
print("first column, pandas default:    ", repr(naive.columns[0]))

with open(RAW / "faostat_qcl_ghana.csv", encoding="utf-8", newline="") as fh:
    print("first column, csv module (utf-8):", repr(next(csv.reader(fh))[0]))

print("item code, pandas default:       ", naive["Item Code (CPC)"].iloc[0], "->", naive["Item Code (CPC)"].dtype)

fao = pd.read_csv(RAW / "faostat_qcl_ghana.csv", encoding="utf-8-sig", dtype={"Item Code (CPC)": "str"})
print("first column, utf-8-sig:         ", repr(fao.columns[0]))
print("item code, read as text:         ", fao["Item Code (CPC)"].iloc[0])

first 3 bytes of file: b'\xef\xbb\xbf'
first column, pandas default:     'Domain Code'
first column, csv module (utf-8): '\ufeffDomain Code'
item code, pandas default:        1520.01 -> float64
first column, utf-8-sig:          'Domain Code'
item code, read as text:          01520.01


Line 
Section 7, FAOSTAT structure: Checked completeness (8 crops × 3 measures × 64 years), blanks, and whether yield = production × 1000 ÷ area. Exactly 1,536 rows, 0 duplicates. All 150 blanks are one block: cowpeas 1961–2010 (not a Project A crop). Yield agrees with production ÷ area in all 462 crop-years (0 disagree by more than 1%) → the three measures are internally consistent, which says nothing about whether they’re accurate; that depends on the flags (Section 8).

In [11]:
print("rows:", len(fao), "   expected 8 x 3 x 64 =", 8 * 3 * 64)
print("duplicate item/element/year:", int(fao.duplicated(["Item", "Element", "Year"]).sum()))
print("unit per element:", fao.dropna(subset=["Unit"]).groupby("Element")["Unit"].agg(lambda s: s.unique().tolist()).to_dict())
print("blanks per column:", fao.isna().sum()[lambda s: s > 0].to_dict())

# Where are the blank values?
fao[fao["Value"].isna()].groupby(["Item", "Element"])["Year"].agg(["min", "max", "count"])

rows: 1536    expected 8 x 3 x 64 = 1536
duplicate item/element/year: 0
unit per element: {'Area harvested': ['ha'], 'Production': ['t'], 'Yield': ['kg/ha']}
blanks per column: {'Unit': 150, 'Value': 150, 'Flag': 150, 'Flag Description': 150, 'Note': 1502}


min   max  count
Item          Element                          
Cow peas, dry Area harvested  1961  2010     50
              Production      1961  2010     50
              Yield           1961  2010     50

In [12]:
# One row per crop-year, one column per measure.
wide = fao.pivot_table(index=["Item", "Year"], columns="Element", values="Value")
implied = wide["Production"] * 1000 / wide["Area harvested"]
gap = (wide["Yield"] - implied).abs() / wide["Yield"]   # 0.01 = 1% apart

print("crop-years compared:", int(gap.notna().sum()), "   disagreeing by more than 1%:", int((gap > 0.01).sum()))

crop-years compared: 462    disagreeing by more than 1%: 0


Section 8, FAOSTAT flags: Checked the source flag of every Project A value, 2016–2023 (A official, E estimated, I imputed, X other organisation). Up to 2021, most years have 12–15 of 15 official; 2017 dips to 7. In 2022 and 2023 only 1 of 15 is official (4 E, 3 I, 7 X) → the most recent production and yield figures are mostly not official. The Flag column must survive cleaning, and any Project A output using 2022–2023 must say so.

In [13]:
A_ITEMS = ["Maize (corn)", "Millet", "Sorghum", "Rice", "Cassava, fresh"]
use = fao[fao["Item"].isin(A_ITEMS) & fao["Year"].between(2006, 2023)]

# For each year: how many of the 15 values (5 crops x 3 measures) carry each flag.
pd.crosstab(use["Year"], use["Flag"].fillna("blank")).loc[2016:]

Flag,A,E,I,X
Year,,,,
2016,15,0,0,0
2017,7,5,2,1
2018,13,1,1,0
2019,12,0,0,3
2020,13,1,0,1
2021,13,1,0,1
2022,1,4,3,7
2023,1,4,3,7



Section 9, Bank of Ghana: Checked which column can serve as a key. Meeting No. 107 appears twice (July 2022 and an August 2022 emergency meeting), so Meeting No. can’t be used. Effective Date can: 0 dates failed to parse, all unique, in date order. 130 decisions, November 2002 to March 2026; prices stop in July 2023, so only the overlap is comparable. The rate changed at 60 of 129 decisions → the file has one row per decision, not per month; Phase 3 carries each rate forward to the next decision and logs that as a modelling step.

In [14]:
bog = pd.read_csv(RAW / "bog_policy_rate.csv")
print(bog.shape)
print(bog.dtypes)

dup = bog[bog["Meeting No."].duplicated(keep=False)]
print("\nmeeting numbers used more than once:", dup["Meeting No."].unique().tolist())
dup

(130, 4)
Meeting No.          int64
MPC Dates              str
Effective Date         str
BOG Policy Rate    float64
dtype: object

meeting numbers used more than once: [107]


,Meeting No.,MPC Dates,Effective Date,BOG Policy Rate
106,107,"July 20 – 22, 2022",25 Jul 2022,19.0
107,107,"August 17, 2022 (Emergency)",17 Aug 2022,22.0


In [15]:
eff = pd.to_datetime(bog["Effective Date"], format="%d %b %Y")   # explicit format: any odd date raises an error
print("dates unparsed:", int(eff.isna().sum()), "   unique:", eff.is_unique, "   in date order:", eff.is_monotonic_increasing)
print("from", eff.min().date(), "to", eff.max().date())

rate = bog["BOG Policy Rate"]
print(f"rate range: {rate.min()}% to {rate.max()}%")
print("decisions that changed the rate:", int((rate.diff().iloc[1:] != 0).sum()), "of", len(rate) - 1)

dates unparsed: 0    unique: True    in date order: True
from 2002-11-21 to 2026-03-18
rate range: 12.5% to 30.0%
decisions that changed the rate: 60 of 129


Section 1, shape, types, blanks: Both files unchanged since acquisition (MD5 match). The price file is 37,765 rows × 16 columns and the markets file 93 × 7; both checked by assert. price, usdprice, latitude and longitude are numbers; ids are integers; the other 10 columns are text (str under pandas 3.0.6). 0 blanks in any column. That doesn’t mean the values are correct (Section 8 found 92 wrong ones) → no cleaning needed for structure.

Section 2, dates: All 37,765 dates are the 15th of the month, January 2006 to July 2023. They’re monthly values stamped at mid-month, not survey days. Every year has 12 months except 2023, which has 7 → the data is monthly at best, so Phase 4 stores date as a month. Any cross-year average by month gets 2023 in January–July only.

Section 3, market identity: 44 markets in the price file. Each market_id has exactly one name, region, district and location; no name uses two ids. All 44 exist in the markets file, with 0 mismatches on any attribute → no identity cleaning needed. 49 of the 93 markets in the markets file have no prices, so Phase 4 loads only the 44 or a Power BI market filter will list empty markets.